# 04 — Fine-tuning: Linear Probe / Full Fine-tune on LIDC-IDRI

Loads the JEPA-pretrained encoder, builds labels from XML annotations,
and fine-tunes a classification head for binary lung-cancer detection.

In [1]:
# ── Cell 1: Project root & sys.path ──────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

print("Project root :", PROJECT_ROOT)
print("src exists   :", (PROJECT_ROOT / "src").exists())

Project root : /home/sreethanu/Downloads/lung_cancer_vljepa
src exists   : True


In [2]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import random
import xml.etree.ElementTree as ET
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm

from src.models.encoder_3d import ViT3DEncoder
from src.models.classification_head import ClassificationHead
from src.utils.config import Config
from src.utils.seed import set_global_seed
from src.utils.losses import FocalLoss
from src.utils.metrics import (
    compute_classification_metrics,
    tune_threshold_from_roc,
    predict_with_threshold,
    print_metrics_summary,
)

print("Imports OK.")

Imports OK.


In [3]:
# ── Cell 3: Config, seed, device ─────────────────────────────────────────────
config_path = PROJECT_ROOT / "configs" / "jepa_config.yaml"
config = Config(str(config_path))

set_global_seed(
    config.get("project.seed", 42),
    deterministic=config.get("project.deterministic", True),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device              :", device)
print("Configured shape    :", config.get("data.input_shape", [1, 96, 96, 96]))

Setting global seed: 42
Enabling deterministic mode (may reduce performance)
Reproducibility configured successfully.
Device              : cuda
Configured shape    : [1, 96, 96, 96]


In [4]:
# ── Cell 4: Instantiate encoder (must match pretrained checkpoint) ────────────
online_encoder = ViT3DEncoder(
    input_size=tuple(config.get("data.input_shape", [1, 96, 96, 96])[1:]),
    patch_size=tuple(config.get("model.encoder.patch_size", [16, 16, 16])),
    embed_dim=config.get("model.encoder.embed_dim", 384),
    depth=config.get("model.encoder.depth", 6),
    num_heads=config.get("model.encoder.num_heads", 6),
).to(device)

print("Num patches :", online_encoder.get_num_patches())

Num patches : 216


In [5]:
# ── Cell 5: Load pretrained JEPA checkpoint ───────────────────────────────────
best_ckpt     = PROJECT_ROOT / "checkpoints" / "pretraining" / "best_model.pth"
fallback_ckpt = PROJECT_ROOT / "checkpoints" / "pretraining" / "jepa_epoch_100.pth"

checkpoint_path = best_ckpt if best_ckpt.exists() else fallback_ckpt
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)

online_encoder.load_state_dict(checkpoint["online_encoder"])
online_encoder.eval()

print(f"Checkpoint keys : {list(checkpoint.keys())}")
print(f"Loaded from     : {checkpoint_path.name}")

sd = checkpoint["online_encoder"]
print("embed_dim   :", sd["patch_embed.proj.bias"].shape[0])
print("num_layers  :", max(int(k.split(".")[2]) for k in sd if k.startswith("transformer.layers.")) + 1)

Checkpoint keys : ['epoch', 'best_val_loss', 'online_encoder', 'target_encoder', 'predictor', 'optimizer']
Loaded from     : best_model.pth
embed_dim   : 384
num_layers  : 6


In [6]:
# ── Cell 6: Parse LIDC-IDRI XML annotations ───────────────────────────────────
ANNOTATION_DIR = PROJECT_ROOT / "data" / "raw" / "annotations" / "tcia-lidc-xml"
PATCH_DIR      = PROJECT_ROOT / "data" / "processed" / "patches"
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"

xml_files = sorted(ANNOTATION_DIR.rglob("*.xml"))
print("XML files found :", len(xml_files))

ns = {"ns": "http://www.nih.gov"}
series_nodule_scores = defaultdict(lambda: defaultdict(list))
series_to_patient    = {}

for xml_file in xml_files:
    try:
        root = ET.parse(xml_file).getroot()
    except Exception:
        continue

    series_elem = root.find(".//ns:SeriesInstanceUid", ns)
    if series_elem is None or not series_elem.text:
        continue
    series_uid = series_elem.text.strip()

    folder_num = xml_file.parent.name
    try:
        patient_id = f"LIDC-IDRI-{int(folder_num):04d}"
        series_to_patient[series_uid] = patient_id
    except ValueError:
        pass

    for nodule in root.findall(".//ns:unblindedReadNodule", ns):
        nid_elem = nodule.find(".//ns:noduleID", ns)
        mal_elem = nodule.find(".//ns:characteristics/ns:malignancy", ns)
        if nid_elem is None or mal_elem is None:
            continue
        if nid_elem.text is None or mal_elem.text is None:
            continue
        try:
            score = float(mal_elem.text)
        except Exception:
            continue
        series_nodule_scores[series_uid][nid_elem.text.strip()].append(score)

print("Series with malignancy scores :", len(series_nodule_scores))
print("Series-to-patient mappings    :", len(series_to_patient))
print("Sample patient_ids            :", list(series_to_patient.values())[:5])

XML files found : 1318
Series with malignancy scores : 883
Series-to-patient mappings    : 1018
Sample patient_ids            : ['LIDC-IDRI-0157', 'LIDC-IDRI-0157', 'LIDC-IDRI-0157', 'LIDC-IDRI-0157', 'LIDC-IDRI-0157']


In [7]:
# ── Cell 7: Build labels_df ───────────────────────────────────────────────────
labels_data        = []
excluded_ambiguous = 0
missing_patches    = 0

for series_uid, nodule_dict in series_nodule_scores.items():
    patient_id = series_to_patient.get(series_uid)
    if patient_id is None:
        missing_patches += 1
        continue

    patch_dir   = PATCH_DIR / patient_id
    patch_files = sorted(patch_dir.glob("*.npz"))
    if len(patch_files) == 0:
        missing_patches += 1
        continue

    nodule_avgs = [
        float(np.mean(scores))
        for scores in nodule_dict.values()
        if len(scores) > 0
    ]
    if len(nodule_avgs) == 0:
        continue

    patient_max = max(nodule_avgs)
    if patient_max >= 4.0:
        label = 1
    elif patient_max <= 2.0:
        label = 0
    else:
        excluded_ambiguous += 1
        continue

    labels_data.append({
        "series_uid"            : series_uid,
        "patient_id"            : patient_id,
        "patient_max_malignancy": patient_max,
        "num_nodules"           : len(nodule_avgs),
        "num_patches"           : len(patch_files),
        "label"                 : label,
    })

labels_df = pd.DataFrame(labels_data).sort_values("series_uid").reset_index(drop=True)

print(f"Labeled series       : {len(labels_df)}")
print(f"Excluded (ambiguous) : {excluded_ambiguous}")
print(f"Missing patches      : {missing_patches}")
print("\nLabel distribution:")
print(labels_df["label"].value_counts())
print("\nSample rows:")
print(labels_df.head())

Labeled series       : 541
Excluded (ambiguous) : 199
Missing patches      : 143

Label distribution:
label
1    479
0     62
Name: count, dtype: int64

Sample rows:
                                          series_uid      patient_id  \
0  1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...  LIDC-IDRI-0188   
1  1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...  LIDC-IDRI-0188   
2  1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...  LIDC-IDRI-0188   
3  1.3.6.1.4.1.14519.5.2.1.6279.6001.102133688497...  LIDC-IDRI-0188   
4  1.3.6.1.4.1.14519.5.2.1.6279.6001.102681962408...  LIDC-IDRI-0186   

   patient_max_malignancy  num_nodules  num_patches  label  
0                     4.0            7           51      1  
1                     4.0            6           51      1  
2                     5.0           16           51      1  
3                     2.0            3           51      0  
4                     5.0            8           21      1  


In [8]:
# ── Cell 8: Save labels.csv & reload ─────────────────────────────────────────
METADATA_DIR = PROCESSED_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
LABELS_CSV = METADATA_DIR / "labels.csv"

labels_df.to_csv(LABELS_CSV, index=False)
print("Saved to :", LABELS_CSV)

metadata = pd.read_csv(LABELS_CSV)
print("Reloaded rows :", len(metadata))
print(metadata["label"].value_counts())

Saved to : /home/sreethanu/Downloads/lung_cancer_vljepa/data/processed/metadata/labels.csv
Reloaded rows : 541
label
1    479
0     62
Name: count, dtype: int64


In [9]:
# ── Cell 9: Class weights ─────────────────────────────────────────────────────
class_counts  = metadata["label"].value_counts().sort_index().to_dict()
total         = sum(class_counts.values())
weights       = [total / (2 * class_counts[c]) for c in sorted(class_counts)]
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print("Class counts       :", class_counts)
print("Balanced CE weights:", class_weights)

Class counts       : {0: 62, 1: 479}
Balanced CE weights: tensor([4.3629, 0.5647], device='cuda:0')


In [10]:
# ── Cell 10: Encoder freeze/unfreeze ─────────────────────────────────────────
train_encoder = bool(config.get("linear_probe.train_encoder", True))

for param in online_encoder.parameters():
    param.requires_grad = train_encoder

trainable = sum(p.numel() for p in online_encoder.parameters() if p.requires_grad)
total_enc = sum(p.numel() for p in online_encoder.parameters())
print(f"Train encoder            : {train_encoder}")
print(f"Encoder trainable params : {trainable:,} / {total_enc:,}")

Train encoder            : True
Encoder trainable params : 12,303,744 / 12,303,744


In [11]:
# ── Cell 11: Build EncoderClassifier model ────────────────────────────────────
classifier = ClassificationHead(
    embed_dim   =config.get("model.encoder.embed_dim", 384),
    hidden_dims =config.get("model.classifier.hidden_dims", [512, 256]),
    num_classes =2,
).to(device)


class EncoderClassifier(nn.Module):
    """Wraps ViT3DEncoder + ClassificationHead into one forward pass."""

    def __init__(self, encoder, head, freeze_encoder=False):
        super().__init__()
        self.encoder        = encoder
        self.head           = head
        self.freeze_encoder = freeze_encoder

    def forward(self, x):
        if self.freeze_encoder:
            with torch.no_grad():
                feats = self.encoder(x)
        else:
            feats = self.encoder(x)
        return self.head(feats)


model = EncoderClassifier(
    online_encoder,
    classifier,
    freeze_encoder=not train_encoder,
).to(device)

print("Model ready.")

Model ready.


In [12]:
# ── Cell 12: Optimizer, loss, scheduler ──────────────────────────────────────
# FIX 1: alpha raised 0.25 → 0.75.
#   alpha=0.25 was under-weighting malignant (label=1, the majority class),
#   which combined with WeightedRandomSampler caused near-zero loss signal
#   and a flat AUC stuck at ~0.5 across all 30 epochs.
# FIX 2: encoder_lr raised 1e-5 → 5e-5.
#   1e-5 was too small to meaningfully update 12M encoder params in 30 epochs.

epochs       = int(config.get("linear_probe.epochs", 30))
encoder_lr   = float(config.get("linear_probe.encoder_lr", 5e-5))   # was 1e-5
head_lr      = float(config.get("linear_probe.head_lr", 5e-4))
weight_decay = float(config.get("linear_probe.weight_decay", 1e-2))

if train_encoder:
    optimizer = torch.optim.AdamW(
        [
            {"params": online_encoder.parameters(), "lr": encoder_lr},
            {"params": classifier.parameters(),     "lr": head_lr},
        ],
        weight_decay=weight_decay,
    )
else:
    optimizer = torch.optim.AdamW(
        classifier.parameters(),
        lr=head_lr,
        weight_decay=weight_decay,
    )

criterion = FocalLoss(alpha=0.75, gamma=2.0)   # was alpha=0.25
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-6,
)

print(f"Epochs       : {epochs}")
print(f"Encoder LR   : {encoder_lr}")
print(f"Head LR      : {head_lr}")
print(f"Loss         : FocalLoss(alpha=0.75, gamma=2.0)")
print(f"Scheduler    : CosineAnnealingLR")

Epochs       : 30
Encoder LR   : 5e-05
Head LR      : 0.0005
Loss         : FocalLoss(alpha=0.75, gamma=2.0)
Scheduler    : CosineAnnealingLR


In [13]:
# ── Cell 13: Dataset class (inline, patient_id fix applied) ───────────────────
class LIDCClassificationDataset(Dataset):
    """
    Series-level dataset: samples one random nodule patch per series per access.
    Returns (volume, label, series_uid) — 3-tuple.
    Patch dirs are named by patient_id, NOT series_uid.
    """

    def __init__(self, processed_dir, metadata, augment=False):
        self.processed_dir = Path(processed_dir)
        self.augment       = augment

        rows    = []
        missing = 0
        for _, row in metadata.reset_index(drop=True).iterrows():
            series_uid = str(row["series_uid"])
            patient_id = str(row["patient_id"])
            label      = int(row["label"])

            patch_dir  = self.processed_dir / "patches" / patient_id
            if not patch_dir.exists():
                missing += 1
                continue

            patch_files = sorted(patch_dir.glob("*.npz"))
            if patch_files:
                rows.append((series_uid, patient_id, label, patch_files))
            else:
                missing += 1

        self.rows = rows
        print(f"  {len(self.rows)} series loaded  |  {missing} had no patches on disk.")

    def __len__(self):
        return len(self.rows)

    @staticmethod
    def _load(patch_file):
        data = np.load(patch_file)
        if "patch" in data:
            return data["patch"]
        if "context" in data:
            return data["context"]
        raise KeyError(f"No 'patch'/'context' key in {patch_file}. Keys: {list(data.keys())}")

    @staticmethod
    def _augment(volume):
        for dim in [1, 2, 3]:
            if random.random() > 0.5:
                volume = torch.flip(volume, [dim])
        k = random.randint(0, 3)
        if k > 0:
            volume = torch.rot90(volume, k, [2, 3])
        if random.random() > 0.5:
            volume = (volume + torch.randn_like(volume) * 0.01).clamp(0, 1)
        return volume

    def __getitem__(self, idx):
        series_uid, patient_id, label, patch_files = self.rows[idx]
        arr    = self._load(random.choice(patch_files))
        volume = torch.tensor(arr, dtype=torch.float32)
        if volume.ndim == 3:
            volume = volume.unsqueeze(0)
        if self.augment:
            volume = self._augment(volume)
        return volume, torch.tensor(label, dtype=torch.long), series_uid


print("LIDCClassificationDataset defined.")

LIDCClassificationDataset defined.


In [14]:
# ── Cell 14: Train/val split + Datasets + DataLoaders ────────────────────────
# FIX: Split on patient_id level, not series_uid level.
# Previously, multiple series from the same patient could appear on both sides
# of the split, causing label leakage and inflated val metrics.

unique_patients = metadata[["patient_id", "label"]].drop_duplicates("patient_id")

train_patients, val_patients = train_test_split(
    unique_patients,
    test_size    =0.20,
    random_state =config.get("project.seed", 42),
    stratify     =unique_patients["label"],
)

train_meta = metadata[metadata["patient_id"].isin(train_patients["patient_id"])].reset_index(drop=True)
val_meta   = metadata[metadata["patient_id"].isin(val_patients["patient_id"])].reset_index(drop=True)

print(f"Train patients : {len(train_patients)}  |  Train series : {len(train_meta)}")
print(f"Val   patients : {len(val_patients)}    |  Val   series : {len(val_meta)}")
print("Train labels :", train_meta["label"].value_counts().to_dict())
print("Val   labels :", val_meta["label"].value_counts().to_dict())

print("\nBuilding train dataset...")
train_dataset = LIDCClassificationDataset(
    processed_dir=str(PROCESSED_DIR),
    metadata     =train_meta,
    augment      =True,
)
print("Building val dataset...")
val_dataset = LIDCClassificationDataset(
    processed_dir=str(PROCESSED_DIR),
    metadata     =val_meta,
    augment      =False,
)

assert len(train_dataset) > 0, "Train dataset is empty!"
assert len(val_dataset)   > 0, "Val dataset is empty!"

print(f"\nTrain dataset : {len(train_dataset)} series")
print(f"Val   dataset : {len(val_dataset)} series")

# ── Weighted sampler to handle class imbalance ────────────────────────────────
train_labels   = [row[2] for row in train_dataset.rows]
class_sample_w = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_labels)
sample_weights = [float(class_sample_w[lbl]) for lbl in train_labels]
sampler        = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

batch_size = int(config.get("data.batch_size", 8))

train_loader = DataLoader(
    train_dataset,
    batch_size  =batch_size,
    sampler     =sampler,
    num_workers =0,
    pin_memory  =True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size  =batch_size,
    shuffle     =False,
    num_workers =0,
    pin_memory  =True,
)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")

Train patients : 4  |  Train series : 457
Val   patients : 1    |  Val   series : 84
Train labels : {1: 401, 0: 56}
Val   labels : {1: 78, 0: 6}

Building train dataset...
  457 series loaded  |  0 had no patches on disk.
Building val dataset...
  84 series loaded  |  0 had no patches on disk.

Train dataset : 457 series
Val   dataset : 84 series

Train batches : 77
Val   batches : 14


In [15]:
# ── Cell 15: Sanity check — one batch ────────────────────────────────────────
vols, lbls, sids = next(iter(train_loader))
print("Batch volume shape :", vols.shape)    # expect (B, 1, 96, 96, 96)
print("Batch labels       :", lbls.tolist())
print("Batch series[0]    :", sids[0])

Batch volume shape : torch.Size([6, 1, 96, 96, 96])
Batch labels       : [1, 1, 1, 0, 1, 0]
Batch series[0]    : 1.3.6.1.4.1.14519.5.2.1.6279.6001.803987517543436570820681016103


In [16]:
# ── Cell 16: Training loop ────────────────────────────────────────────────────
scaler         = GradScaler()
best_state     = None
best_val_auc   = -1.0
best_threshold = 0.5
history        = []

CKPT_DIR = PROJECT_ROOT / "checkpoints" / "finetuning"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

for epoch in range(epochs):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0

    for volumes, labels, _ in tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{epochs} [train]", leave=False):
        volumes = volumes.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            logits = model(volumes)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    scheduler.step()
    avg_train_loss = total_loss / max(len(train_loader), 1)

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    val_probs  = []
    val_labels = []

    with torch.no_grad():
        for volumes, labels, _ in val_loader:
            volumes = volumes.to(device, non_blocking=True)
            logits  = model(volumes)
            probs   = torch.softmax(logits, dim=1)[:, 1]
            val_probs.extend(probs.cpu().numpy())
            val_labels.extend(labels.numpy())

    val_probs  = np.asarray(val_probs)
    val_labels = np.asarray(val_labels)

    tune            = tune_threshold_from_roc(val_labels, val_probs, min_sensitivity=0.80)
    epoch_threshold = tune["threshold"]
    val_preds       = predict_with_threshold(val_probs, epoch_threshold)
    val_metrics     = compute_classification_metrics(val_labels, val_preds, val_probs)

    history.append({
        "epoch"          : epoch + 1,
        "train_loss"     : avg_train_loss,
        "val_auc"        : val_metrics.get("roc_auc", 0.0),
        "val_sensitivity": val_metrics.get("sensitivity", 0.0),
        "val_specificity": val_metrics.get("specificity", 0.0),
        "threshold"      : epoch_threshold,
    })

    if val_metrics.get("roc_auc", 0.0) > best_val_auc:
        best_val_auc   = val_metrics["roc_auc"]
        best_threshold = epoch_threshold
        best_state     = {
            "epoch"     : epoch + 1,
            "encoder"   : online_encoder.state_dict(),
            "classifier": classifier.state_dict(),
            "val_auc"   : best_val_auc,
            "threshold" : best_threshold,
        }
        torch.save(best_state, CKPT_DIR / "best_finetune.pth")

    print(
        f"Epoch {epoch+1:02d}/{epochs} "
        f"| Loss {avg_train_loss:.4f} "
        f"| AUC {val_metrics.get('roc_auc', 0.0):.4f} "
        f"| Sens {val_metrics.get('sensitivity', 0.0):.4f} "
        f"| Spec {val_metrics.get('specificity', 0.0):.4f} "
        f"| Thr {epoch_threshold:.3f}"
    )

if best_state is not None:
    online_encoder.load_state_dict(best_state["encoder"])
    classifier.load_state_dict(best_state["classifier"])

print(f"\nBest val AUC  : {best_val_auc:.4f}")
print(f"Best threshold: {best_threshold:.4f}")
print(f"Checkpoint saved to : {CKPT_DIR / 'best_finetune.pth'}")

Epoch 01/30 | Loss 0.1321 | AUC 0.7222 | Sens 0.9359 | Spec 0.1667 | Thr 0.501


Epoch 02/30 | Loss 0.1311 | AUC 0.5235 | Sens 0.8333 | Spec 0.0000 | Thr 0.501


Epoch 03/30 | Loss 0.1310 | AUC 0.5737 | Sens 1.0000 | Spec 0.0000 | Thr 0.507


Epoch 04/30 | Loss 0.1301 | AUC 0.5171 | Sens 0.8205 | Spec 0.3333 | Thr 0.496


Epoch 05/30 | Loss 0.1300 | AUC 0.3440 | Sens 0.8077 | Spec 0.1667 | Thr 0.502


...

Epoch 26/30 | Loss 0.1276 | AUC 0.4915 | Sens 0.8333 | Spec 0.3333 | Thr 0.510


Epoch 27/30 | Loss 0.1298 | AUC 0.6015 | Sens 0.8974 | Spec 0.1667 | Thr 0.509


Epoch 28/30 | Loss 0.1279 | AUC 0.5833 | Sens 0.8462 | Spec 0.0000 | Thr 0.508


Epoch 29/30 | Loss 0.1290 | AUC 0.5481 | Sens 0.8974 | Spec 0.1667 | Thr 0.508


Epoch 30/30 | Loss 0.1300 | AUC 0.5288 | Sens 0.8077 | Spec 0.1667 | Thr 0.514

Best val AUC  : 0.7222
Best threshold: 0.5006
Checkpoint saved to : /home/sreethanu/Downloads/lung_cancer_vljepa/checkpoints/finetuning/best_finetune.pth


In [17]:
# ── Cell 17: Patch-level evaluation on val set ────────────────────────────────
model.eval()

all_probs  = []
all_labels = []
all_series = []

with torch.no_grad():
    for volumes, labels, series_uids in val_loader:
        volumes = volumes.to(device, non_blocking=True)
        logits  = model(volumes)
        probs   = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_series.extend(series_uids)

all_probs  = np.asarray(all_probs)
all_labels = np.asarray(all_labels)
all_preds  = predict_with_threshold(all_probs, best_threshold)

patch_metrics = compute_classification_metrics(all_labels, all_preds, all_probs)

print("=" * 50)
print("Patch-Level Metrics (Val set)")
print("=" * 50)
print_metrics_summary(patch_metrics)

print("\nSample patch predictions (first 10):")
for i in range(min(10, len(all_probs))):
    print(
        f"  series={all_series[i][:40]}... "
        f"true={all_labels[i]} prob={all_probs[i]:.4f} pred={all_preds[i]}"
    )

Patch-Level Metrics (Val set)

LUNG CANCER CLASSIFICATION RESULTS
ACCURACY            : 0.9286
BALANCED_ACCURACY   : 0.5000
ROC_AUC             : 0.4840
PR_AUC              : 0.9311
SENSITIVITY         : 1.0000
SPECIFICITY         : 0.0000
PRECISION           : 0.9286
RECALL              : 1.0000
F1                  : 0.9630
MCC                 : 0.0000
PPV                 : 0.9286
NPV                 : 0.0000


Sample patch predictions (first 10):
  series=1.3.6.1.4.1.14519.5.2.1.6279.6001.105495... true=1 prob=0.5137 pred=1
  series=1.3.6.1.4.1.14519.5.2.1.6279.6001.113679... true=1 prob=0.5081 pred=1
  series=1.3.6.1.4.1.14519.5.2.1.6279.6001.124663... true=1 prob=0.5081 pred=1
  series=1.3.6.1.4.1.14519.5.2.1.6279.6001.126631... true=1 prob=0.5141 pred=1
  series=1.3.6.1.4.1.14519.5.2.1.6279.6001.128023... true=1 prob=0.5032 pred=1
  series=1.3.6.1.4.1.14519.5.2.1.6279.6001.129982... true=0 prob=0.5139 pred=1
  series=1.3.6.1.4.1.14519.5.2.1.6279.6001.130794... true=1 prob=0.5147 p

In [18]:
# ── Cell 18: Scan-level evaluation (max-pool over all patches) ────────────────
# FIX: Added seen_patients deduplication guard.
# Previously, multiple series rows sharing the same patient_id caused the same
# physical patch folder to be evaluated multiple times, producing repeated
# identical predictions and making scan-level metrics meaningless.

model.eval()

scan_preds    = []
scan_labels   = []
scan_probs    = []
scan_examples = []
seen_patients = set()                           # ← deduplication guard

with torch.no_grad():
    for _, row in val_meta.iterrows():
        patient_id = str(row["patient_id"])
        if patient_id in seen_patients:         # ← skip already-evaluated patients
            continue
        seen_patients.add(patient_id)

        true_label  = int(row["label"])
        series_path = PATCH_DIR / patient_id
        patch_files = sorted(series_path.glob("*.npz"))
        if len(patch_files) == 0:
            continue

        probs_list = []
        for patch_file in patch_files:
            data = np.load(patch_file)
            if "patch" in data:
                arr = data["patch"]
            elif "context" in data:
                arr = data["context"]
            else:
                continue

            volume = torch.tensor(arr, dtype=torch.float32)
            if volume.ndim == 3:
                volume = volume.unsqueeze(0)
            volume = volume.unsqueeze(0).to(device)

            logits = model(volume)
            prob   = torch.softmax(logits, dim=1)[0, 1].item()
            probs_list.append(prob)

        if len(probs_list) == 0:
            continue

        scan_prob = max(probs_list)
        pred      = int(scan_prob >= best_threshold)

        scan_preds.append(pred)
        scan_labels.append(true_label)
        scan_probs.append(scan_prob)

        if len(scan_examples) < 10:
            scan_examples.append((patient_id, true_label, scan_prob, pred, len(probs_list)))

scan_metrics = compute_classification_metrics(
    np.array(scan_labels),
    np.array(scan_preds),
    np.array(scan_probs),
)

print("=" * 50)
print("Scan-Level Aggregated Metrics (MAX probability)")
print(f"Evaluated : {len(seen_patients)} unique patients")
print("=" * 50)
print_metrics_summary(scan_metrics)

print("\nSample scan predictions (first 10):")
for patient_id, true_label, scan_prob, pred, n_patches in scan_examples:
    print(
        f"  {patient_id} | true={true_label} "
        f"| patches={n_patches} "
        f"| prob={scan_prob:.4f} | pred={pred}"
    )

Scan-Level Aggregated Metrics (MAX probability)
Evaluated : 1 unique patients

LUNG CANCER CLASSIFICATION RESULTS
ACCURACY            : 1.0000
BALANCED_ACCURACY   : 1.0000
ROC_AUC             : nan
PR_AUC              : 1.0000
SENSITIVITY         : 1.0000
SPECIFICITY         : 0.0000
PRECISION           : 1.0000
RECALL              : 1.0000
F1                  : 1.0000
MCC                 : 0.0000
PPV                 : 1.0000
NPV                 : 0.0000


Sample scan predictions (first 10):
  LIDC-IDRI-0185 | true=1 | patches=39 | prob=0.5170 | pred=1


In [19]:
# ── Cell 19: Training history summary ────────────────────────────────────────
history_df = pd.DataFrame(history)
print(history_df.to_string(index=False))

print(f"\nBest epoch   : {history_df.loc[history_df['val_auc'].idxmax(), 'epoch']}")
print(f"Best val AUC : {best_val_auc:.4f}")
print(f"Threshold    : {best_threshold:.4f}")

 epoch  train_loss  val_auc  val_sensitivity  val_specificity  threshold
     1    0.132131 0.722222         0.935897         0.166667   0.500552
     2    0.131116 0.523504         0.833333         0.000000   0.501468
     3    0.131039 0.573718         1.000000         0.000000   0.507406
     4    0.130062 0.517094         0.820513         0.333333   0.496496
     5    0.130010 0.344017         0.807692         0.166667   0.501847
     6    0.130333 0.672009         0.871795         0.333333   0.509230
     7    0.129690 0.391026         0.846154         0.166667   0.528613
     8    0.131080 0.325855         0.807692         0.000000   0.514709
     9    0.129917 0.613248         0.884615         0.000000   0.497047
    10    0.129795 0.673077         0.897436         0.000000   0.487477
    11    0.130480 0.564103         0.846154         0.166667   0.520823
    12    0.129929 0.462607         0.846154         0.000000   0.500375
    13    0.130360 0.697650         0.820513       